# Case Citation Trend — citations per case per year

`case_citation.parquet` collapses the citation history into four windows. This keeps the
history: one row per (case, citing year), so any window — or a decay curve, or an age profile —
can be rebuilt downstream without re-scanning the edge list.

## Output
`Case law/output/case_citation_trend.parquet`

| column | meaning |
|---|---|
| `case_id` | the cited case |
| `decision_year` | its decision year |
| `cite_year` | decision year of the citing case |
| `yrs_since_decision` | `cite_year - decision_year`, always `>= 0` |
| `C` | citations received from that year |

Only (case, year) pairs with at least one citation appear — the table is sparse, not a dense
case × year grid, which is what keeps it to a manageable size. A case never cited is absent
entirely; `case_citation.parquet` is where its zero lives.

Sorted by `case_id, cite_year`, so a per-case scan touches one contiguous run.

In [1]:
%%time
import os, sys, gc
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Case law')
import cl_common as cl
OUT_FP = cl.out('case_citation_trend.parquet')
cl.preflight('case_citation_trend')

c_from, c_to, year, uni = cl.load_graph()
yF, yT = year[c_from], year[c_to]
ok = (yF > 0) & (yT > 0) & (yF >= yT)
print(f'edges {len(c_from):,}  ->  counted {int(ok.sum()):,}')

# Group by (cited case, citing year). Done as one integer key so the group-by is a single
# sort rather than a two-column hash over 47.5M rows.
cited = c_to[ok].astype(np.int64)
cyear = yF[ok].astype(np.int64)
YEAR0 = 1600
key = cited * 512 + (cyear - YEAR0)           # 512 > 2030-1600, so no collision
uk, cnt = np.unique(key, return_counts=True)
del key, cited, cyear; gc.collect()
print(f'(case, cite_year) pairs: {len(uk):,}')

case law : /project/jevans/Dawoon/Science of Science/Case law
output   : /project/jevans/Dawoon/Science of Science/Case law/output
cache    : /project/jevans/Dawoon/Science of Science/Case law/cache

  case_citation_trend         OK
graph cache present: /project/jevans/Dawoon/Science of Science/Case law/cache/case_graph.npz
edges 47,519,638  ->  counted 47,519,638
(case, cite_year) pairs: 25,816,136


In [2]:
%%time
code = (uk // 512).astype(np.int64)
cy   = (uk % 512).astype(np.int32) + YEAR0
trend = pd.DataFrame({
    'case_id':            uni[code],
    'decision_year':      year[code].astype(np.int32),
    'cite_year':          cy,
    'yrs_since_decision': (cy - year[code]).astype(np.int32),
    'C':                  cnt.astype(np.int32),
}).sort_values(['case_id', 'cite_year']).reset_index(drop=True)
assert (trend.yrs_since_decision >= 0).all(), 'a citation cannot predate the decision'
trend.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(trend):,} rows, {os.path.getsize(OUT_FP)/1e6:.0f} MB)')
print(f'  cases represented {trend.case_id.nunique():,}   '
      f'total citations {int(trend.C.sum()):,}')
display(trend.head(6))
print('\nmean citations by age (first 15 years):')
display(trend.groupby('yrs_since_decision')['C'].agg(['mean', 'sum']).head(15).round(3))

WROTE /project/jevans/Dawoon/Science of Science/Case law/output/case_citation_trend.parquet  (25,816,136 rows, 101 MB)
  cases represented 3,787,861   total citations 47,519,638


,case_id,decision_year,cite_year,yrs_since_decision,C
0,1,1976,1987,11,1
1,1,1976,1995,19,1
2,1,1976,1996,20,1
3,2,1976,1981,5,2
4,3,1977,1989,12,2
5,3,1977,1993,16,1



mean citations by age (first 15 years):


,mean,sum
yrs_since_decision,,
0,1.969,1535628
1,2.330,3509146
2,2.249,3234050
3,2.168,2818155
4,2.105,2488259
5,2.050,2229278
6,2.010,2026833
7,1.975,1851763
8,1.943,1705208
